# Capstone 4 — MCP Tool Pack for the LMS

You will build an MCP server that exposes the Spark10K LMS data and admin actions, then drive it from Claude Desktop or Claude Code in a live demo. By the end, a natural-language prompt like *"enrol moaaz in the Agentic AI course and email him a welcome"* should execute end-to-end.

## Phase 0 — Concept Recall

1. Why does MCP separate **resources** from **tools**? Give an LMS example of each.
2. Sketch the threat model for `delete_user(user_id)`. What controls do you add?
3. Why is per-call auth scoping safer than per-server auth?

*(5 pts each.)*

In [ ]:
# Phase 1 — Setup
# pip install "mcp[cli]" httpx pydantic
#
# Project layout suggested:
#   lms_mcp/
#     server.py            <- this notebook code, exported
#     tools/
#       courses.py
#       enrolment.py
#       certificates.py
#     auth.py

from mcp.server.fastmcp import FastMCP
import httpx, os, logging

LMS_API = os.getenv("LMS_API", "https://lms.spark10k.com/api")
log = logging.getLogger("lms-mcp")

mcp = FastMCP("lms")


In [ ]:
# Phase 2 — Resources (read-only, the model can pull these as context)

@mcp.resource("lms://courses")
def list_courses() -> str:
    """All available courses with id, title, day count."""
    r = httpx.get(f"{LMS_API}/days", headers=_auth())
    r.raise_for_status()
    return r.text

@mcp.resource("lms://progress/{user_id}")
def user_progress(user_id: str) -> str:
    """A users per-day progress."""
    r = httpx.get(f"{LMS_API}/admin/users/{user_id}/progress", headers=_auth())
    r.raise_for_status()
    return r.text

def _auth():
    # TODO 1: fetch a SCOPED token for the calling user — never your master key.
    return {"Authorization": f"Bearer {os.environ['LMS_USER_TOKEN']}"}


In [ ]:
# Phase 3 — Tools (mutating actions)

from pydantic import BaseModel, EmailStr

class EnrolArgs(BaseModel):
    user_email: EmailStr
    course_id: str
    notify: bool = True

@mcp.tool()
def enrol_user(args: EnrolArgs) -> dict:
    """Enrol a user in a course. Idempotent: re-enrolling is a no-op."""
    # TODO 2: look up user by email; 404 -> typed error, not exception.
    # TODO 3: POST /admin/companies/{cid}/access -- existing endpoint.
    # TODO 4: if notify, POST /admin/email/send.
    # TODO 5: return {"status": "enrolled" | "already_enrolled", "user_id": ...}
    raise NotImplementedError

@mcp.tool()
def issue_certificate(user_id: str, course_id: str, dry_run: bool = False) -> dict:
    """Issue a certificate. dry_run=True returns what WOULD happen, no write."""
    # TODO 6: validate eligibility (progress == 100%) BEFORE writing.
    # TODO 7: when dry_run, return preview JSON only.
    raise NotImplementedError


In [ ]:
# Phase 4 — Prompts (curated templates surfaced to the host)

@mcp.prompt()
def weekly_admin_review() -> str:
    """Run a weekly admin review across enrolments, completions, escalations."""
    return (
        "Use the lms://courses and lms://progress resources to compile:\n"
        "1. New enrolments this week\n"
        "2. Users who finished a course but lack a certificate\n"
        "3. Stalled users (no activity 7+ days)\n"
        "For each stalled user, draft a re-engagement email but DO NOT send it."
    )


In [ ]:
# Phase 5 — Audit log (mandatory)

import json, time, uuid

def audit(tool, args, user, outcome, error=None):
    log.info(json.dumps({
        "ts": time.time(),
        "call_id": str(uuid.uuid4()),
        "tool": tool,
        "user": user,
        "args": args,
        "outcome": outcome,
        "error": error,
    }))

# TODO 8: wrap every @mcp.tool with this audit logger.
# TODO 9: alert if a single user issues >20 mutating tool calls in 60s (abuse signal).


## Phase 6 — Wire it up to a host

1. Run the server: `python -m lms_mcp.server`
2. Add to Claude Desktop config:
   ```json
   {
     "mcpServers": {
       "lms": {
         "command": "python",
         "args": ["-m", "lms_mcp.server"],
         "env": { "LMS_USER_TOKEN": "..." }
       }
     }
   }
   ```
3. Open Claude Desktop. You should see the tools appear.
4. Run the demo task: **"Enrol jane@example.com in the Agentic AI course and prepare a welcome email."**

Record a 60-second screen capture of the end-to-end flow. Submit the video file.

## Phase 7 — Submission checklist (rubric / 100)

- [ ] **25 pts** — Server passes the official MCP Inspector test suite (`mcp dev server.py`).
- [ ] **20 pts** — At least 4 tools, 2 resources, 1 prompt, all with descriptive schemas.
- [ ] **20 pts** — Per-call auth scoping verified with a test that proves user A cannot mutate user Bs data.
- [ ] **15 pts** — Audit log captures every call with `call_id, tool, user, args, outcome`.
- [ ] **20 pts** — Live demo video shows a 3-step admin task completed end-to-end through Claude.

### What you will have *learned by doing*
- That tools are an API design problem, not a prompt-engineering problem.
- That auth, idempotency, and audit are non-negotiable when an LLM is the caller.
- That MCP collapses the integration tax across every AI host you might support.